# Milestone 3 experiment: do artifacts appear with model size? (Google Colab)

**Question:** do high-norm artifact tokens appear in small ViTs trained from scratch, or only once models are
large enough? And do registers change anything below that size?

For every run listed in step 2, this notebook:
1. **trains** it (50 epochs on Tiny-ImageNet, same recipe as the ViT-Tiny runs). Finished runs are skipped instantly;
2. **measures** artifacts in the trained model (token norms after every block, outliers, register use, attention maps);
3. packs the small result files into a zip to download.

| Run | Model | Registers | Approx. time on a T4 |
|---|---|---|---|
| `tiny_reg0`, `tiny_reg4` | ViT-Tiny (5.4M) | 0 / 4 | already trained: only measured (~5 min) |
| `small_reg0`, `small_reg4` | ViT-Small (21.7M) | 0 / 4 | ~2 h each |
| `base_reg0`, `base_reg4` | ViT-Base (85.8M) | 0 / 4 | ~3–4 h each (optional) |

**Several Google accounts:** each account trains different runs at the same time. Training progress is saved to
*that account's* Google Drive, so always continue a run on the same account. The finished results are small
files in the downloaded zip, so they can be combined afterwards.

**Before running:** `Runtime → Change runtime type → T4 GPU`, set `RUNS` in step 2, then `Runtime → Run all`.
If Colab disconnects, run all again: training resumes from the last finished epoch.

## 0. Setup: get the latest code

In [ ]:
import os

REPO_URL = "https://github.com/recommended131225435/vit-registers-reproduction.git"

if os.path.exists("../src") and not os.path.exists("src"):  # opened from inside notebooks/
    %cd ..
if not os.path.exists("src"):                                  # fresh Colab session
    !git clone -q {REPO_URL} repo
    %cd repo
else:                                                          # same session: take the latest code
    !git fetch -q && git reset -q --hard origin/main
!pip install -q -r requirements.txt

import torch
from IPython.display import Image, Markdown, display
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4!)")

## 1. Connect Google Drive
Trained models are saved here after every epoch. Click **Allow** when asked.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/vit-registers-runs"
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs are saved in", RUN_DIR)

## 2. Choose the runs for THIS account
Main account (the one that trained ViT-Tiny): `["tiny_reg0", "tiny_reg4", "small_reg0"]`.
Second account: `["small_reg4"]`. Optional third/fourth account: `["base_reg0"]`, `["base_reg4"]`.

In [ ]:
RUNS = ["tiny_reg0", "tiny_reg4", "small_reg0"]   # <-- edit for this account

SIZES = {"tiny", "small", "base"}
for run in RUNS:
    size, reg = run.split("_reg")
    assert size in SIZES and reg in {"0", "4"}, f"bad run name: {run}"
print("This account runs:", RUNS)

## 3. Train (finished runs print `Resuming from epoch 50` and are skipped)

In [ ]:
for run in RUNS:
    size, reg = run.split("_reg")
    print(f"===== {run} =====")
    !python -m src.train --size {size} --registers {reg} --epochs 50 --resume --out {RUN_DIR}/{run}

## 4. Measure artifacts in each trained model

In [ ]:
for run in RUNS:
    !python -m scripts.analyse_run --run {RUN_DIR}/{run}

## 5. Export the ViT-Tiny models for the deployment app (main account only; skipped otherwise)

In [ ]:
for run in ["tiny_reg0", "tiny_reg4"]:
    if run in RUNS:
        !python -m scripts.export_model --run {RUN_DIR}/{run} --out app/weights/vit_{run}.pt

## 6. Table and figures from the runs measured in this session

In [ ]:
!python -m scripts.summarise_experiment
display(Markdown(open("results/experiment/summary.md").read().split("![")[0]))
for name in ["scaling", "layer_norms", "norm_histograms", "accuracy_registers", "attention_maps"]:
    display(Image(f"results/figures/experiment_{name}.png"))

## 7. Download the results
Small files only (no training checkpoints). Send this zip in; one zip per account.

In [ ]:
import glob
paths = [f"results/experiment/{r}" for r in RUNS] + [f"results/runs/{r}" for r in RUNS if os.path.exists(f"results/runs/{r}")]
paths += glob.glob("app/weights/*.pt")
name = "experiment_" + "_".join(RUNS) + ".zip"
targets = " ".join(paths)
!zip -qr {name} {targets}
from google.colab import files
files.download(name)